# Sesión 1 — Primer Análisis de Datos Clínicos
## Perfil hematológico canino: exploración inicial con pandas

**Autora:** Ana Isabel Álvarez  
**Contexto clínico:** Análisis exploratorio de hemogramas de perros adultos  
**Herramientas:** Python 3.10 · pandas · numpy · JupyterLab

In [ ]:
# Las importaciones van siempre al inicio del notebook
# Son equivalentes a los "import" de Java, pero más simples

import pandas as pd      # pd es el alias estándar — verás esto en TODO el mundo de datos
import numpy as np       # np es el alias estándar para numpy
import matplotlib.pyplot as plt  # plt es el alias estándar para graficar

# Esta línea le dice a Jupyter que muestre los gráficos dentro del notebook
%matplotlib inline

# Verificar versiones — útil para reproducibilidad del análisis
print(f"pandas: {pd.__version__}")
print(f"numpy:  {np.__version__}")
print("Entorno listo ✓")

## 1. El dataset: hemogramas caninos

Trabajamos con un conjunto de resultados hematológicos de perros adultos atendidos 
en una clínica veterinaria de referencia. Los datos incluyen el hemograma completo 
con interpretación clínica.

Los rangos de referencia usados son los establecidos por el laboratorio para caninos adultos.

In [ ]:
# En esta sesión creamos el dataset directamente en Python
# A partir de la sesión 2 aprenderemos a cargar archivos reales (CSV, Excel)

# Un diccionario en Python es equivalente a un HashMap en Java:
# llaves (strings) mapeadas a valores (en este caso, listas)

datos_hemograma = {
    # Identificación del paciente
    'paciente_id':      ['C001', 'C002', 'C003', 'C004', 'C005',
                         'C006', 'C007', 'C008', 'C009', 'C010'],
    'raza':             ['Labrador', 'Beagle', 'Pastor Alemán', 'Bulldog', 'Golden',
                         'Schnauzer', 'Poodle', 'Boxer', 'Dálmata', 'Husky'],
    'edad_años':        [3, 7, 5, 4, 2, 9, 6, 8, 3, 5],
    'peso_kg':          [28.5, 12.0, 35.2, 22.1, 30.0, 8.5, 6.2, 29.8, 24.5, 27.3],
    'sexo':             ['M', 'H', 'M', 'M', 'H', 'H', 'M', 'H', 'M', 'H'],

    # Serie roja
    'eritrocitos_mill_uL':  [6.2, 4.8, 7.1, 3.9, 6.5, 5.2, 5.8, 4.1, 6.8, 5.5],
    'hemoglobina_g_dL':     [15.1, 11.8, 17.2, 9.5, 16.0, 12.5, 14.2, 10.2, 16.5, 13.8],
    'hematocrito_pct':      [45.2, 35.4, 51.8, 28.6, 48.0, 37.5, 42.6, 30.8, 49.5, 41.4],
    'VCM_fL':               [72.9, 73.8, 72.9, 73.3, 73.8, 72.1, 73.4, 75.1, 72.8, 75.3],
    'HCM_pg':               [24.4, 24.6, 24.2, 24.4, 24.6, 24.0, 24.5, 24.9, 24.3, 25.1],
    'CHCM_g_dL':            [33.4, 33.3, 33.2, 33.2, 33.3, 33.3, 33.3, 33.1, 33.3, 33.3],

    # Serie blanca
    'leucocitos_mil_uL':    [8.5, 12.3, 7.2, 18.6, 9.1, 11.4, 8.8, 15.2, 7.9, 10.3],
    'neutrofilos_pct':      [68, 75, 65, 82, 70, 78, 67, 80, 66, 72],
    'linfocitos_pct':       [22, 16, 25, 10, 20, 14, 24, 12, 25, 18],
    'monocitos_pct':        [6,  5,  6,  5,  6,  5,  6,  5,  6,  6],
    'eosinofilos_pct':      [3,  3,  3,  2,  3,  2,  2,  2,  2,  3],
    'basofilos_pct':        [1,  1,  1,  1,  1,  1,  1,  1,  1,  1],

    # Plaquetas
    'plaquetas_mil_uL':     [245, 189, 310, 98, 278, 220, 265, 142, 290, 235],

    # Interpretación clínica
    'diagnostico':          ['Normal', 'Anemia leve', 'Normal', 'Anemia moderada + Leucocitosis',
                             'Normal', 'Normal', 'Normal', 'Anemia leve + Leucocitosis',
                             'Normal', 'Normal']
}

# Convertir el diccionario a DataFrame — esta es la estructura central de pandas
# Equivale a una tabla de base de datos: filas son registros, columnas son variables
df = pd.DataFrame(datos_hemograma)

# Confirmar que se creó correctamente
print(f"Dataset creado: {df.shape[0]} pacientes, {df.shape[1]} variables")

## 2. Primera inspección del dataset

El primer paso de cualquier análisis es conocer los datos antes de tocarlos.
Esto es equivalente a revisar la hoja de trabajo antes de empezar a analizar muestras.

In [ ]:
# .head() muestra las primeras filas — por defecto 5
# Es lo primero que ejecutas siempre que cargas un dataset nuevo
df.head()

In [ ]:
# .shape devuelve una tupla (filas, columnas)
# En Java sería equivalente a array.length para cada dimensión
print(f"Filas (pacientes):  {df.shape[0]}")
print(f"Columnas (variables): {df.shape[1]}")

In [ ]:
# .info() es el método más útil para una primera inspección
# Muestra: nombre de columna, cantidad de valores no nulos y tipo de dato
# Dtype object = texto/string | int64 = entero | float64 = decimal
df.info()

In [ ]:
# .dtypes muestra solo los tipos de datos por columna
# Importante: pandas usa sus propios tipos, no los de Python base
print(df.dtypes)

In [ ]:
# .describe() calcula estadísticas descriptivas automáticamente
# Solo para columnas numéricas: count, mean, std, min, cuartiles, max
# En laboratorio: te dice instantáneamente si hay valores imposibles (negativos, extremos)

df.describe().round(2)  # .round(2) = redondear a 2 decimales

## 3. Acceso a columnas y filas — la base de todo análisis

En pandas hay dos formas principales de seleccionar datos.
Dominarlas es fundamental antes de cualquier otra operación.

In [ ]:
# Acceder a una columna: df['nombre_columna']
# Esto devuelve una Series — piénsala como una columna de Excel con índice

hemoglobina = df['hemoglobina_g_dL']
print(type(hemoglobina))   # <class 'pandas.core.series.Series'>
print(hemoglobina)

In [ ]:
# Acceder a múltiples columnas: df[['col1', 'col2', 'col3']]
# Nota los dobles corchetes — el exterior es el operador de pandas,
# el interior es una lista de Python con los nombres de columnas

serie_roja = df[['paciente_id', 'eritrocitos_mill_uL', 'hemoglobina_g_dL', 
                  'hematocrito_pct', 'diagnostico']]
serie_roja

In [ ]:
# Filtrar filas por condición — esto es muy diferente a Java
# En Python/pandas, una condición sobre una columna devuelve una Serie de True/False
# Esa Serie se usa como máscara para filtrar el DataFrame

# Ejemplo clínico: identificar pacientes con anemia (Hto < 37%)
# El rango de referencia para caninos adultos es 37–55%

mascara_anemia = df['hematocrito_pct'] < 37.0
print("Máscara booleana:")
print(mascara_anemia)

print("\nPacientes con hematocrito bajo:")
df[mascara_anemia][['paciente_id', 'raza', 'hematocrito_pct', 
                     'hemoglobina_g_dL', 'diagnostico']]

In [ ]:
# Combinar condiciones con & (AND) y | (OR)
# Importante: cada condición va entre paréntesis — en Java esto no es obligatorio,
# en pandas SÍ lo es porque el orden de operaciones es diferente

# Caso clínico: leucocitosis (> 12,000/µL) con desviación a la izquierda
# aproximada (neutrófilos > 75%)
leucocitosis_con_neutrofilia = (
    (df['leucocitos_mil_uL'] > 12.0) & 
    (df['neutrofilos_pct'] > 75)
)

print("Pacientes con leucocitosis + neutrofilia:")
df[leucocitosis_con_neutrofilia][['paciente_id', 'raza', 
                                   'leucocitos_mil_uL', 
                                   'neutrofilos_pct', 'diagnostico']]

## 4. Estadística descriptiva básica por columna

pandas tiene métodos estadísticos integrados que operan sobre columnas enteras.
No necesitas escribir un bucle para calcular el promedio de 1000 valores.

In [ ]:
# Estadísticas básicas sobre hemoglobina
hgb = df['hemoglobina_g_dL']

print("=== HEMOGLOBINA (g/dL) ===")
print(f"Media:          {hgb.mean():.2f}")
print(f"Mediana:        {hgb.median():.2f}")
print(f"Desv. estándar: {hgb.std():.2f}")
print(f"Mínimo:         {hgb.min():.2f}")
print(f"Máximo:         {hgb.max():.2f}")
print(f"Rango:          {hgb.max() - hgb.min():.2f}")

# El coeficiente de variación (CV%) es clave en control de calidad de laboratorio
cv = (hgb.std() / hgb.mean()) * 100
print(f"CV%:            {cv:.1f}%")

In [ ]:
# Calcular estadísticas para múltiples analitos a la vez
# .agg() aplica múltiples funciones simultáneamente

analitos = ['eritrocitos_mill_uL', 'hemoglobina_g_dL', 'hematocrito_pct',
            'leucocitos_mil_uL', 'plaquetas_mil_uL']

resumen = df[analitos].agg(['mean', 'std', 'min', 'max']).round(2)
print("Resumen estadístico de la serie roja, leucocitos y plaquetas:")
resumen

In [ ]:
# Conteo de categorías — fundamental para variables cualitativas
# ¿Cuántos pacientes hay por diagnóstico?

print("Distribución de diagnósticos:")
print(df['diagnostico'].value_counts())

print("\nDistribución por sexo:")
print(df['sexo'].value_counts())

## 5. Primera visualización — histograma de hemoglobina

Una visualización simple pero directamente interpretable desde la clínica.
El objetivo hoy es familiarizarse con la sintaxis básica de matplotlib.
En el Módulo 4 profundizaremos en visualización avanzada.

In [ ]:
# Estructura básica de un gráfico en matplotlib:
# 1. Crear la figura y los ejes
# 2. Dibujar en los ejes
# 3. Agregar etiquetas y títulos
# 4. Mostrar

fig, ax = plt.subplots(figsize=(8, 5))  
# fig = la hoja de papel | ax = el área donde se dibuja
# figsize = (ancho, alto) en pulgadas

# Histograma de distribución de hemoglobina
ax.hist(df['hemoglobina_g_dL'], 
        bins=6,              # número de barras
        color='steelblue',   
        edgecolor='white',   # borde de las barras
        alpha=0.85)          # transparencia leve

# Línea vertical para el límite inferior de referencia (caninos: 12 g/dL)
ax.axvline(x=12.0, color='red', linestyle='--', linewidth=1.5, 
           label='Límite inferior ref. (12 g/dL)')

# Línea vertical para el límite superior de referencia (caninos: 18 g/dL)
ax.axvline(x=18.0, color='orange', linestyle='--', linewidth=1.5,
           label='Límite superior ref. (18 g/dL)')

# Etiquetas — siempre incluir unidades en laboratorio
ax.set_xlabel('Hemoglobina (g/dL)', fontsize=12)
ax.set_ylabel('Número de pacientes', fontsize=12)
ax.set_title('Distribución de hemoglobina en pacientes caninos\n(n=10)', fontsize=13)
ax.legend()

plt.tight_layout()  # ajusta márgenes automáticamente
plt.show()

## 6. Guardar el dataset limpio

Buena práctica: siempre exportar los datos procesados al final del análisis.

In [ ]:
import os

# Definir la ruta de salida relativa al notebook
ruta_datos = '../datos/'

# Crear la carpeta si no existe (equivalente a mkdir -p)
os.makedirs(ruta_datos, exist_ok=True)

# Exportar a CSV — el formato más universal para datos tabulares
ruta_archivo = os.path.join(ruta_datos, 'hemograma_canino_sesion01.csv')
df.to_csv(ruta_archivo, index=False)  
# index=False: no guardar el índice numérico de pandas como columna

print(f"Dataset guardado en: {ruta_archivo}")
print(f"Filas exportadas: {len(df)}")

In [ ]:
# Paso 1: definir la condición de trombocitopenia leve
# Rango: plaquetas >= 100 mil/µL Y plaquetas < 200 mil/µL
trombocitopenia_leve = (
    (df['plaquetas_mil_uL'] >= 100) & 
    (df['plaquetas_mil_uL'] < 200)
)

# Paso 2: filtrar y seleccionar columnas relevantes
pacientes_trombocitopenia = df[trombocitopenia_leve][
    ['paciente_id', 'raza', 'plaquetas_mil_uL', 'diagnostico']
]

print("Pacientes con trombocitopenia leve:")
print(pacientes_trombocitopenia)

# Paso 3: media de plaquetas en el subgrupo
media_plaquetas_subgrupo = pacientes_trombocitopenia['plaquetas_mil_uL'].mean()
print(f"\nMedia de plaquetas en este subgrupo: {media_plaquetas_subgrupo:.1f} mil/µL")